In [ ]:
# =============================================================================
# PS3 CALIBRATION -- do device-native labels agree with ServiceNow?
#
# PURPOSE. PS3 is moving from ServiceNow availability events onto the canonical
# OOS spine (docs/OOS_EVENT_CONTRACT.md), with device-native targets and
# ServiceNow as enrichment / validation / audit trail (Option 3).
#
# Before any of that is built, four questions need answering WITH DATA:
#
#   Q1 COVERAGE   -- what share of OOS events have a ServiceNow record at all,
#                    per fleet? (Validators are expected near zero -- that is
#                    the gap Option 3 exists to close.)
#   Q2 ROOT CAUSE -- where both exist, does the device's COMPONENT_TYPE_NAME /
#                    component_subsystem agree with SN root_cause_category /
#                    affected_component?
#   Q3 SEVERITY   -- does event_type_severity / severity / event_priority agree
#                    with SN failure_level_label / AE_FAILURE_LEVEL?
#   Q4 TIMING     -- is the device row actually available AT failure time?
#                    Measured as EDW_INSERTED_DTM - EVENT_DTM. This is the
#                    assumption the whole at-failure-time design rests on, and
#                    it has been asserted but never measured.
#
# HOW TO READ THE RESULT.
#   High agreement  -> native labels stand alone. Option 3 is safe. SN becomes
#                      audit trail + backfill.
#   Low agreement   -> the two are measuring DIFFERENT THINGS, which is not a
#                      failure: it means SN captures what the device cannot
#                      self-report, and the disagreement itself is signal.
#   Either way this is informative. There is no "bad" outcome, only a design
#   that follows the data instead of preceding it.
#
# READ-ONLY. Creates no table, writes no parquet, mutates nothing.
# Run in the PS3 Studio space. Do NOT run alongside a PS5 run on one instance.
# =============================================================================

GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
SCHEME      = "s3a"
def SILVER(t): return f"{SCHEME}://{GOLD_BUCKET}/chicago/silver/{t}"

RUN_DATE        = "2026-04-11"     # ODS watermark, matches PS2/PS5
TELEMETRY_START = "2024-01-01"
SCOPE           = ["TVM", "GATE", "VALIDATOR"]

print("=" * 78)
print("PS3 CALIBRATION -- device-native vs ServiceNow")
print(f"window {TELEMETRY_START} -> {RUN_DATE} | scope {SCOPE}")
print("=" * 78)


In [ ]:
# ── Cell 0 — Java for local Spark ───────────────────────────────────────────
# The PS3 space has no pyspark and may have no JDK. This is the SAME bootstrap the
# PS5 spine builder uses -- known to work in this account's Studio images.
# If a JDK is already present this is a no-op in a couple of seconds.
import os, glob, subprocess

def _find_java_home():
    cands = []
    for base in ("/opt/conda", "/usr/lib/jvm", "/usr/local", "/usr"):
        cands += glob.glob(f"{base}/**/bin/java", recursive=True)
    return os.path.dirname(os.path.dirname(cands[0])) if cands else None

jh = os.environ.get("JAVA_HOME") or _find_java_home()
if not jh:
    print("No JDK found - installing OpenJDK 17 via conda (~1-2 min)...")
    subprocess.run("conda install -y -c conda-forge openjdk=17", shell=True)
    jh = _find_java_home()
assert jh, "Java not found. Spark cannot start without a JDK."
os.environ["JAVA_HOME"] = jh
os.environ["PATH"] = f"{jh}/bin:" + os.environ.get("PATH", "")
print("JAVA_HOME =", jh)
subprocess.run(["java", "-version"])


In [ ]:
# ── Cell 1 — Spark bootstrap (reuse active session, else local S3A) ─────────
# Driver memory is set for an ml.m5.2xlarge (32 GiB): 22g driver leaves headroom
# for the JVM overhead and the Python process. On an ml.r5.2xlarge (64 GiB) you
# can raise it to 48g. Do NOT set it above ~70% of instance RAM.
#
# device_event_enriched is 228.8M rows / 21.2 GB and UNPARTITIONED, so the
# transit_day filter is a full scan with predicate pushdown rather than partition
# elimination. Column pruning is what saves us: the calibration selects ~17 of 76
# columns, so roughly 17/76 of 21.2 GB is actually read. This is I/O-bound, not
# memory-bound -- if it feels slow, that is the scan, not the instance.
import os, sys, subprocess, importlib

def _pip(*pkgs):
    try: ip = get_ipython()
    except NameError: ip = None
    if ip is not None: ip.run_line_magic("pip", "install -q " + " ".join(pkgs))
    else: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    importlib.invalidate_caches()

try:
    import pyspark  # noqa
except ImportError:
    _pip("pyspark==3.5.9")

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

HADOOP_AWS_PACKAGE = "org.apache.hadoop:hadoop-aws:3.3.4"   # matches Spark 3.5 / Hadoop 3.3.x
DRIVER_MEM = os.environ.get("PS3_DRIVER_MEM", "22g")

spark = SparkSession.getActiveSession()
if spark is None:
    os.environ.setdefault("PYSPARK_SUBMIT_ARGS", f"--driver-memory {DRIVER_MEM} pyspark-shell")
    spark = (SparkSession.builder.appName("PS3_Calibration")
             .master("local[*]")
             .config("spark.driver.memory", DRIVER_MEM)
             .config("spark.driver.maxResultSize", "4g")
             .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
             .config("spark.sql.adaptive.enabled", "true")
             .config("spark.sql.parquet.filterPushdown", "true")
             .config("spark.jars.packages", HADOOP_AWS_PACKAGE)
             .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                     "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
             .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
             .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version, "| master:", spark.sparkContext.master,
      "| driver mem:", DRIVER_MEM)


In [ ]:
# -----------------------------------------------------------------------------
# 0. SCHEMA CONFIRMATION -- fail here, not 20 minutes in
# -----------------------------------------------------------------------------
dee = spark.read.parquet(SILVER("device_event_enriched"))
irc = spark.read.parquet(SILVER("incident_root_cause"))
dim = spark.read.parquet(SILVER("dim_device"))

NEED_DEE = ["DEVICE_ID", "DEVICE_KEY", "EVENT_DTM", "transit_day", "mars_device_category",
            "is_hardware_oos_event", "EVENT_STATE_TYPE_NAME",
            "COMPONENT_TYPE_NAME", "component_subsystem", "COMPONENT_SERIAL_NBR",
            "EVENT_TYPE_NAME", "EVENT_TYPE_DESC",
            "event_type_severity", "severity", "event_priority",
            "EDW_INSERTED_DTM", "SOURCE_INSERTED_DTM"]
NEED_IRC = ["device_id", "transit_day", "root_cause_category", "affected_component",
            "failure_level_label", "AE_FAILURE_LEVEL", "AE_FAULT_DESCRIPTION",
            "is_chargeable", "is_device_fault", "AE_START_DTM", "mars_device_category"]

missing_dee = [c for c in NEED_DEE if c not in dee.columns]
missing_irc = [c for c in NEED_IRC if c not in irc.columns]
print(f"\ndevice_event_enriched : {len(dee.columns)} cols | missing {missing_dee or 'none'}")
print(f"incident_root_cause   : {len(irc.columns)} cols | missing {missing_irc or 'none'}")
assert not missing_dee, f"device_event_enriched missing {missing_dee}"
assert not missing_irc, f"incident_root_cause missing {missing_irc}"


In [ ]:
# -----------------------------------------------------------------------------
# 1. THE CANONICAL OOS SPINE  (docs/OOS_EVENT_CONTRACT.md, two clauses only)
#    Identical to PS1's attach_hardware_oos_label(), device_category as a param.
#    UPPER(TRIM(...)) per contract 3.2 -- PS1's bare == "Set" is not robust.
# -----------------------------------------------------------------------------
current = dim.where(F.col("is_current") == True).select("DEVICE_KEY").distinct()

oos = (
    dee.alias("e")
    .join(F.broadcast(current).alias("d"), F.col("e.DEVICE_KEY") == F.col("d.DEVICE_KEY"))
    .where(F.col("e.is_hardware_oos_event") == True)
    .where(F.upper(F.trim(F.col("e.EVENT_STATE_TYPE_NAME").cast("string"))) == "SET")
    .where(F.upper(F.col("e.mars_device_category").cast("string")).isin(*SCOPE))
    .where((F.col("e.transit_day") >= F.lit(TELEMETRY_START))
           & (F.col("e.transit_day") <= F.lit(RUN_DATE)))
    .select(
        F.col("e.DEVICE_ID").cast("string").alias("device_id"),
        F.col("e.transit_day").alias("transit_day"),
        F.col("e.EVENT_DTM").alias("event_dtm"),
        F.upper(F.col("e.mars_device_category").cast("string")).alias("fleet"),
        F.col("e.COMPONENT_TYPE_NAME").cast("string").alias("dev_component"),
        F.col("e.component_subsystem").cast("string").alias("dev_subsystem"),
        F.col("e.EVENT_TYPE_NAME").cast("string").alias("dev_event_type"),
        F.col("e.event_type_severity").cast("double").alias("dev_severity_num"),
        F.col("e.severity").cast("string").alias("dev_severity_lbl"),
        F.col("e.event_priority").cast("int").alias("dev_priority"),
        F.col("e.EDW_INSERTED_DTM").alias("edw_inserted_dtm"),
        F.col("e.SOURCE_INSERTED_DTM").alias("source_inserted_dtm"),
    )
).persist()

n_oos = oos.count()
print(f"\nOOS spine: {n_oos:,} events")
oos.groupBy("fleet").agg(
    F.count("*").alias("events"),
    F.countDistinct("device_id").alias("devices"),
    F.countDistinct(F.struct("device_id", "transit_day")).alias("device_days"),
).orderBy("fleet").show(truncate=False)

# device-day grain, per contract (PS1/PS5 use calendar day)
oos_day = (oos.groupBy("device_id", "transit_day", "fleet")
              .agg(F.first("dev_component", True).alias("dev_component"),
                   F.first("dev_subsystem", True).alias("dev_subsystem"),
                   F.first("dev_event_type", True).alias("dev_event_type"),
                   F.max("dev_severity_num").alias("dev_severity_num"),
                   F.first("dev_severity_lbl", True).alias("dev_severity_lbl"),
                   F.max("dev_priority").alias("dev_priority"))).persist()


In [ ]:
# -----------------------------------------------------------------------------
# Q4 TIMING -- is the device row available AT failure time?
#    The entire at-failure-time design assumes yes. Measure it.
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("Q4  TIMING -- lag from EVENT_DTM to the row landing in EDW")
print("=" * 78)
lag = (oos.withColumn("edw_lag_min",
            (F.unix_timestamp("edw_inserted_dtm") - F.unix_timestamp("event_dtm")) / 60.0)
          .where(F.col("edw_lag_min").isNotNull()))
lag.groupBy("fleet").agg(
    F.round(F.expr("percentile_approx(edw_lag_min, 0.50)"), 1).alias("p50_min"),
    F.round(F.expr("percentile_approx(edw_lag_min, 0.90)"), 1).alias("p90_min"),
    F.round(F.expr("percentile_approx(edw_lag_min, 0.99)"), 1).alias("p99_min"),
    F.round(F.avg(F.when(F.col("edw_lag_min") <= 60, 1.0).otherwise(0.0)) * 100, 1).alias("pct_within_1h"),
    F.round(F.avg(F.when(F.col("edw_lag_min") <= 1440, 1.0).otherwise(0.0)) * 100, 1).alias("pct_within_24h"),
).orderBy("fleet").show(truncate=False)
print("  If p90 is minutes-to-hours, at-failure-time scoring is viable.")
print("  If it is days, PS3 cannot score at failure time regardless of design.")


In [ ]:
# -----------------------------------------------------------------------------
# Q1 COVERAGE -- what share of OOS device-days have a ServiceNow record?
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("Q1  COVERAGE -- OOS device-days with a matching ServiceNow record")
print("=" * 78)
sn = (irc
      .where((F.col("transit_day") >= F.lit(TELEMETRY_START))
             & (F.col("transit_day") <= F.lit(RUN_DATE)))
      .select(
          F.col("device_id").cast("string").alias("device_id"),
          F.col("transit_day").alias("transit_day"),
          F.col("root_cause_category").cast("string").alias("sn_root_cause"),
          F.col("affected_component").cast("string").alias("sn_component"),
          F.col("failure_level_label").cast("string").alias("sn_failure_level"),
          F.col("AE_FAILURE_LEVEL").cast("double").alias("sn_failure_level_num"),
          F.col("AE_FAULT_DESCRIPTION").cast("string").alias("sn_fault_desc"),
          F.col("is_chargeable").alias("sn_is_chargeable"),
      )
      .dropDuplicates(["device_id", "transit_day"])).persist()

joined = oos_day.join(sn, ["device_id", "transit_day"], "left").persist()
joined.groupBy("fleet").agg(
    F.count("*").alias("oos_device_days"),
    F.sum(F.when(F.col("sn_root_cause").isNotNull(), 1).otherwise(0)).alias("with_sn_root_cause"),
    F.round(F.avg(F.when(F.col("sn_root_cause").isNotNull(), 1.0).otherwise(0.0)) * 100, 1).alias("pct_sn_cover"),
    F.sum(F.when(F.col("dev_component").isNotNull(), 1).otherwise(0)).alias("with_dev_component"),
    F.round(F.avg(F.when(F.col("dev_component").isNotNull(), 1.0).otherwise(0.0)) * 100, 1).alias("pct_dev_cover"),
).orderBy("fleet").show(truncate=False)
print("  pct_dev_cover is the prize: label coverage Option 3 delivers.")
print("  pct_sn_cover is what PS3 has today. The gap is the validator blind spot.")


In [ ]:
# -----------------------------------------------------------------------------
# Q2 ROOT CAUSE agreement
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("Q2  ROOT CAUSE -- device COMPONENT_TYPE_NAME/subsystem vs SN root_cause_category")
print("=" * 78)
both = joined.where(F.col("sn_root_cause").isNotNull() & F.col("dev_component").isNotNull()).persist()
print(f"  overlap rows (both labels present): {both.count():,}\n")

for dev_col, sn_col, label in [("dev_component",  "sn_component",  "device COMPONENT_TYPE_NAME  vs  SN affected_component"),
                                ("dev_component",  "sn_root_cause", "device COMPONENT_TYPE_NAME  vs  SN root_cause_category"),
                                ("dev_subsystem",  "sn_root_cause", "device component_subsystem  vs  SN root_cause_category")]:
    d = both.where(F.col(sn_col).isNotNull())
    n = d.count()
    if not n:
        print(f"  {label}: no overlap rows\n"); continue
    exact = d.where(F.upper(F.trim(F.col(dev_col))) == F.upper(F.trim(F.col(sn_col)))).count()
    print(f"  {label}")
    print(f"    rows={n:,}  exact-match={exact:,} ({exact/n*100:.1f}%)")
    print("    top pairings:")
    (d.groupBy(F.upper(F.trim(F.col(dev_col))).alias("device"),
               F.upper(F.trim(F.col(sn_col))).alias("servicenow"))
       .count().orderBy(F.desc("count")).show(12, truncate=False))


In [ ]:
# -----------------------------------------------------------------------------
# Q3 SEVERITY agreement
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("Q3  SEVERITY -- device severity signals vs SN failure_level_label")
print("=" * 78)
sev = joined.where(F.col("sn_failure_level").isNotNull())
n_sev = sev.count()
print(f"  rows with SN failure_level_label: {n_sev:,}\n")
if n_sev:
    print("  device event_type_severity distribution per SN failure_level_label:")
    (sev.groupBy("sn_failure_level")
        .agg(F.count("*").alias("n"),
             F.round(F.avg("dev_severity_num"), 2).alias("mean_dev_sev"),
             F.round(F.expr("percentile_approx(dev_severity_num, 0.5)"), 1).alias("med_dev_sev"),
             F.round(F.avg("dev_priority"), 2).alias("mean_dev_priority"))
        .orderBy(F.desc("n")).show(15, truncate=False))
    print("  device severity LABEL vs SN failure_level_label:")
    (sev.groupBy("dev_severity_lbl", "sn_failure_level").count()
        .orderBy(F.desc("count")).show(15, truncate=False))
    print("  MONOTONICITY CHECK: if mean_dev_sev orders the same way as the")
    print("  business severity of the SN labels, the device signal is usable")
    print("  as a severity target even when the class names do not match.")


In [ ]:
# =============================================================================
# Q5 -- HOW OFTEN IS SERVICENOW UNINFORMATIVE WHERE THE DEVICE IS NOT?
#
# The top-12 pairings suggested SN says UNKNOWN against a named device subsystem
# in roughly half the overlap. "Roughly, from the top 12" is not a number you
# design on. This measures it exactly, and measures the reverse case too --
# because if the device is ALSO frequently generic, the argument weakens.
#
# Uses `joined` from the Q1 cell, already persisted. No re-scan.
# =============================================================================
print("\n" + "=" * 78)
print("Q5  UNINFORMATIVE-LABEL ANALYSIS -- who actually knows the cause?")
print("=" * 78)

# Values that carry no diagnostic content on either side.
SN_UNINFORMATIVE  = ["UNKNOWN", "OTHER", "NONE", "N/A", "NA", ""]
# Device subsystems that name the whole device rather than a component.
DEV_UNINFORMATIVE = ["SYSTEM", "DEV", "DEVICE", "OTHER", "UNKNOWN", "NONE", ""]

_sn_u  = F.upper(F.trim(F.coalesce(F.col("sn_root_cause"), F.lit(""))))
_dev_u = F.upper(F.trim(F.coalesce(F.col("dev_subsystem"), F.lit(""))))

lab = (joined
       .withColumn("sn_informative",
                   F.when(F.col("sn_root_cause").isNull(), F.lit(False))
                    .when(_sn_u.isin(*SN_UNINFORMATIVE), F.lit(False))
                    .otherwise(F.lit(True)))
       .withColumn("dev_informative",
                   F.when(F.col("dev_subsystem").isNull(), F.lit(False))
                    .when(_dev_u.isin(*DEV_UNINFORMATIVE), F.lit(False))
                    .otherwise(F.lit(True))))

# --- (a) across ALL OOS device-days: who has a usable label at all? ----------
print("\n  (a) ALL OOS device-days -- usable label availability")
lab.groupBy("fleet").agg(
    F.count("*").alias("oos_device_days"),
    F.round(F.avg(F.col("dev_informative").cast("double")) * 100, 1).alias("pct_dev_informative"),
    F.round(F.avg(F.col("sn_informative").cast("double")) * 100, 1).alias("pct_sn_informative"),
).orderBy("fleet").show(truncate=False)
print("      pct_sn_informative is the REAL trainable share PS3 has today --")
print("      lower than raw SN coverage, because UNKNOWN rows are not labels.")

# --- (b) on the OVERLAP only: the four-way breakdown -------------------------
print("\n  (b) OVERLAP only (both tables have a row) -- who knows what")
ov = lab.where(F.col("sn_root_cause").isNotNull())
n_ov = ov.count()
print(f"      overlap rows: {n_ov:,}\n")
if n_ov:
    (ov.groupBy("dev_informative", "sn_informative")
       .agg(F.count("*").alias("n"),
            F.round(F.count("*") / F.lit(n_ov) * 100, 1).alias("pct"))
       .orderBy(F.desc("n")).show(truncate=False))
    dev_only = ov.where(F.col("dev_informative") & ~F.col("sn_informative")).count()
    sn_only  = ov.where(~F.col("dev_informative") & F.col("sn_informative")).count()
    both_i   = ov.where(F.col("dev_informative") & F.col("sn_informative")).count()
    neither  = ov.where(~F.col("dev_informative") & ~F.col("sn_informative")).count()
    print(f"      device knows, SN says UNKNOWN : {dev_only:,} ({dev_only/n_ov*100:.1f}%)")
    print(f"      SN knows, device is generic   : {sn_only:,} ({sn_only/n_ov*100:.1f}%)")
    print(f"      both informative              : {both_i:,} ({both_i/n_ov*100:.1f}%)")
    print(f"      neither informative           : {neither:,} ({neither/n_ov*100:.1f}%)")
    print()
    print("      READ: if device-knows-SN-doesn't >> SN-knows-device-doesn't, then")
    print("      ServiceNow is NOT a superior ground truth being approximated --")
    print("      the device is the better source and SN is the enrichment.")
    print("      If they are comparable, both are needed and neither dominates.")

# --- (c) exactly which SN values are the uninformative ones ------------------
print("\n  (c) SN root_cause_category value distribution on the overlap")
(ov.groupBy(F.upper(F.trim(F.coalesce(F.col("sn_root_cause"), F.lit("<null>")))).alias("sn_root_cause"))
   .agg(F.count("*").alias("n"))
   .withColumn("pct", F.round(F.col("n") / F.lit(n_ov) * 100, 1))
   .orderBy(F.desc("n")).show(25, truncate=False))

# --- (d) and the device side, for symmetry ----------------------------------
print("  (d) device component_subsystem distribution on the overlap")
(ov.groupBy(F.upper(F.trim(F.coalesce(F.col("dev_subsystem"), F.lit("<null>")))).alias("dev_subsystem"))
   .agg(F.count("*").alias("n"))
   .withColumn("pct", F.round(F.col("n") / F.lit(n_ov) * 100, 1))
   .orderBy(F.desc("n")).show(25, truncate=False))


In [ ]:
# =============================================================================
# Q6 -- IS SEVERITY DERIVABLE FROM WHICH SUBSYSTEM FAILED?
#
# The device severity COLUMNS are dead (severity == 'INFO' for all 68,121 rows,
# event_type_severity 10.0-or-NULL, event_priority 1.68-2.00). So severity
# cannot come from a device severity field.
#
# But SN's failure_level_label describes FUNCTIONAL IMPACT -- PURCHASE_CARD,
# ALL_PURCHASE, ALL_FUNCTIONS -- i.e. what stopped working. That is plausibly a
# CONSEQUENCE of which subsystem failed, not an independent measurement. A card
# reader failing implies the card purchase path is down.
#
# If each subsystem maps predominantly to one failure level, severity is
# derivable at failure time and the severity head can stay device-native.
# If the distribution is flat, it cannot, and severity stays post-work-order.
#
# The deciding statistic is MODAL SHARE: for each subsystem, what fraction of
# its rows land in its single most common failure level.
# =============================================================================
print("\n" + "=" * 78)
print("Q6  SEVERITY DERIVABILITY -- component_subsystem x failure_level_label")
print("=" * 78)

sev6 = (joined
        .where(F.col("sn_failure_level").isNotNull() & F.col("dev_subsystem").isNotNull())
        .withColumn("subsystem", F.upper(F.trim(F.col("dev_subsystem"))))
        .withColumn("failure_level", F.upper(F.trim(F.col("sn_failure_level")))))
n6 = sev6.count()
print(f"  rows with BOTH subsystem and failure_level: {n6:,}\n")

if n6:
    # --- raw crosstab -------------------------------------------------------
    print("  (a) raw crosstab (rows=subsystem, cols=failure_level)")
    sev6.groupBy("subsystem").pivot("failure_level").count().na.fill(0) \
        .orderBy("subsystem").show(30, truncate=False)

    # --- modal share: THE decision metric -----------------------------------
    print("  (b) MODAL SHARE per subsystem -- the deciding number")
    w_sub = Window.partitionBy("subsystem")
    w_rank = Window.partitionBy("subsystem").orderBy(F.desc("n"))
    modal = (sev6.groupBy("subsystem", "failure_level").agg(F.count("*").alias("n"))
                 .withColumn("subsystem_total", F.sum("n").over(w_sub))
                 .withColumn("rk", F.row_number().over(w_rank))
                 .where(F.col("rk") == 1)
                 .withColumn("modal_share_pct",
                             F.round(F.col("n") / F.col("subsystem_total") * 100, 1))
                 .select("subsystem", "subsystem_total",
                         F.col("failure_level").alias("modal_failure_level"),
                         F.col("n").alias("modal_n"), "modal_share_pct")
                 .orderBy(F.desc("subsystem_total")))
    modal.show(30, truncate=False)

    # --- weighted average modal share = accuracy of a pure lookup table -----
    agg = modal.agg(
        F.sum("modal_n").alias("modal_total"),
        F.sum("subsystem_total").alias("grand_total")).collect()[0]
    lookup_acc = agg["modal_total"] / agg["grand_total"] * 100
    n_sub = modal.count()

    # --- baseline: always predict the single most common failure level ------
    top_overall = (sev6.groupBy("failure_level").agg(F.count("*").alias("n"))
                       .orderBy(F.desc("n")).first())
    base_acc = top_overall["n"] / n6 * 100

    print("\n  (c) VERDICT")
    print(f"      subsystems observed            : {n_sub}")
    print(f"      subsystem -> modal level accuracy: {lookup_acc:.1f}%")
    print(f"      majority-class baseline         : {base_acc:.1f}%  "
          f"(always predict {top_overall['failure_level']})")
    print(f"      lift over baseline              : {lookup_acc - base_acc:+.1f} pts")
    print()
    if lookup_acc >= 70 and (lookup_acc - base_acc) >= 15:
        print("      => STRONG. Subsystem largely determines functional impact.")
        print("         Severity can be a DEVICE-NATIVE head, scored at failure time,")
        print("         with the mapping published as a lookup and a model on top.")
    elif (lookup_acc - base_acc) >= 8:
        print("      => PARTIAL. Subsystem carries real severity signal but does not")
        print("         determine it. A device-native severity head is viable WITH")
        print("         additional features (device type, event type, time of day).")
    else:
        print("      => WEAK. Subsystem does not determine functional impact.")
        print("         Severity must stay ServiceNow-derived, which means the")
        print("         severity head is a POST-WORK-ORDER model even though the")
        print("         root-cause head scores at failure time. Two different SLAs.")

    # --- does adding fleet help? severity is device-type dependent ----------
    print("\n  (d) same, split by fleet -- a card reader on a TVM and on a gate")
    print("      do not have the same functional impact")
    w_sf   = Window.partitionBy("fleet", "subsystem")
    w_sfr  = Window.partitionBy("fleet", "subsystem").orderBy(F.desc("n"))
    modal_f = (sev6.groupBy("fleet", "subsystem", "failure_level").agg(F.count("*").alias("n"))
                   .withColumn("tot", F.sum("n").over(w_sf))
                   .withColumn("rk", F.row_number().over(w_sfr))
                   .where(F.col("rk") == 1))
    aggf = modal_f.agg(F.sum("n").alias("m"), F.sum("tot").alias("t")).collect()[0]
    lookup_acc_f = aggf["m"] / aggf["t"] * 100
    print(f"      (fleet, subsystem) -> modal level accuracy: {lookup_acc_f:.1f}%")
    print(f"      gain from adding fleet: {lookup_acc_f - lookup_acc:+.1f} pts")
    modal_f.select("fleet", "subsystem", F.col("failure_level").alias("modal_level"),
                   "n", "tot",
                   F.round(F.col("n") / F.col("tot") * 100, 1).alias("modal_share_pct")) \
           .orderBy("fleet", F.desc("tot")).show(40, truncate=False)

print("\n" + "=" * 78)
print("Q5/Q6 COMPLETE")
print("=" * 78)


In [ ]:
# =============================================================================
# Q6b -- SEVERITY DERIVABILITY, MEASURED ON THE REAL IMPACT CLASSES ONLY
#
# WHY THIS EXISTS. Q6 returned 47.1% accuracy vs a 43.5% baseline (+3.6 pts) and
# concluded WEAK. That verdict is arithmetically correct and, I think, answers
# the wrong question -- because 71.7% of failure_level_label is not severity:
#
#     BACK_OFFICE_OR_UNKNOWN   29,628   43.5%   a catch-all, not a severity
#     FULLY_FUNCTIONAL         19,211   28.2%   the device still WORKS
#     ------------------------------------------------------------------
#     PURCHASE_CARD            11,796   17.3%   real functional impact
#     ALL_PURCHASE              4,286    6.3%   real functional impact
#     ALL_FUNCTIONS             2,424    3.6%   real functional impact
#     PURCHASE_PRODUCT            776    1.1%   real functional impact
#
# Asking whether component_subsystem predicts a target that is mostly "unknown"
# and "nothing was wrong" is not a fair test of whether it predicts SEVERITY.
#
# HAND-CALC FROM THE Q6 CROSSTAB (to be confirmed or refuted by this cell):
#   restricted to the four impact classes -> accuracy 71.1%, baseline 61.2%,
#   lift +9.9 pts. That crosses the PARTIAL threshold and approaches STRONG.
#
# THIS CELL IS THE CHECK. If it disagrees with those numbers, believe the cell.
#
# Uses `joined`, already persisted. No re-scan.
# =============================================================================
print("\n" + "=" * 78)
print("Q6b  SEVERITY DERIVABILITY -- real functional-impact classes only")
print("=" * 78)

# The classes that describe what actually stopped working. Everything else is
# either a catch-all or an explicit statement that nothing failed.
IMPACT_CLASSES = ["PURCHASE_CARD", "ALL_PURCHASE", "ALL_FUNCTIONS", "PURCHASE_PRODUCT"]
NON_IMPACT     = ["BACK_OFFICE_OR_UNKNOWN", "FULLY_FUNCTIONAL"]

base = (joined
        .where(F.col("sn_failure_level").isNotNull() & F.col("dev_subsystem").isNotNull())
        .withColumn("subsystem",     F.upper(F.trim(F.col("dev_subsystem"))))
        .withColumn("failure_level", F.upper(F.trim(F.col("sn_failure_level")))))
n_all = base.count()

sev = base.where(F.col("failure_level").isin(*IMPACT_CLASSES)).persist()
n6b = sev.count()

print(f"  all rows with subsystem + failure_level : {n_all:,}")
print(f"  restricted to real impact classes       : {n6b:,} ({n6b/max(n_all,1)*100:.1f}%)")
print(f"  dropped as non-severity                 : {n_all - n6b:,} "
      f"({(n_all-n6b)/max(n_all,1)*100:.1f}%)  {NON_IMPACT}\n")

if n6b == 0:
    print("  !! no rows survived the filter -- check the class spellings against Q6(a)")
else:
    # --- modal share per subsystem -----------------------------------------
    print("  (a) MODAL SHARE per subsystem, impact classes only")
    w_sub  = Window.partitionBy("subsystem")
    w_rank = Window.partitionBy("subsystem").orderBy(F.desc("n"))
    modal = (sev.groupBy("subsystem", "failure_level").agg(F.count("*").alias("n"))
                .withColumn("subsystem_total", F.sum("n").over(w_sub))
                .withColumn("rk", F.row_number().over(w_rank))
                .where(F.col("rk") == 1)
                .withColumn("modal_share_pct",
                            F.round(F.col("n") / F.col("subsystem_total") * 100, 1))
                .select("subsystem", "subsystem_total",
                        F.col("failure_level").alias("modal_impact_level"),
                        F.col("n").alias("modal_n"), "modal_share_pct")
                .orderBy(F.desc("subsystem_total")))
    modal.show(30, truncate=False)

    agg = modal.agg(F.sum("modal_n").alias("m"), F.sum("subsystem_total").alias("t")).collect()[0]
    lookup_acc = agg["m"] / agg["t"] * 100
    top = sev.groupBy("failure_level").agg(F.count("*").alias("n")).orderBy(F.desc("n")).first()
    base_acc = top["n"] / n6b * 100
    lift = lookup_acc - base_acc

    # --- the same, adding fleet --------------------------------------------
    w_sf  = Window.partitionBy("fleet", "subsystem")
    w_sfr = Window.partitionBy("fleet", "subsystem").orderBy(F.desc("n"))
    modal_f = (sev.groupBy("fleet", "subsystem", "failure_level").agg(F.count("*").alias("n"))
                  .withColumn("tot", F.sum("n").over(w_sf))
                  .withColumn("rk", F.row_number().over(w_sfr))
                  .where(F.col("rk") == 1))
    aggf = modal_f.agg(F.sum("n").alias("m"), F.sum("tot").alias("t")).collect()[0]
    acc_f = aggf["m"] / aggf["t"] * 100

    # --- class balance: a 4-class problem needs usable minority classes ----
    print("  (b) impact-class balance (a rare class the model never sees is not learnable)")
    (sev.groupBy("failure_level").agg(F.count("*").alias("n"))
        .withColumn("pct", F.round(F.col("n") / F.lit(n6b) * 100, 1))
        .orderBy(F.desc("n")).show(truncate=False))

    # --- verdict ------------------------------------------------------------
    print("  (c) VERDICT")
    print(f"      rows                              : {n6b:,}")
    print(f"      subsystem -> modal impact accuracy: {lookup_acc:.1f}%")
    print(f"      majority-class baseline           : {base_acc:.1f}%  "
          f"(always predict {top['failure_level']})")
    print(f"      lift over baseline                : {lift:+.1f} pts")
    print(f"      (fleet, subsystem) accuracy       : {acc_f:.1f}%  "
          f"({acc_f - lookup_acc:+.1f} from adding fleet)")
    print()
    print(f"      hand-calc from the Q6 crosstab was 71.1% / 61.2% / +9.9 pts")
    print(f"      this cell says              {lookup_acc:.1f}% / {base_acc:.1f}% / {lift:+.1f} pts")
    _delta = abs(lookup_acc - 71.1)
    print(f"      agreement: {'CONFIRMED' if _delta < 2 else 'DIVERGES -- trust this cell, not the hand-calc'}"
          f" (delta {_delta:.1f} pts)")
    print()
    if lookup_acc >= 70 and lift >= 15:
        print("      => STRONG. Subsystem largely determines functional impact.")
        print("         Severity is a DEVICE-NATIVE head, scored at failure time.")
        print("         PS3 stays ONE product with ONE SLA.")
    elif lift >= 8:
        print("      => PARTIAL -- and good enough. Subsystem carries real severity")
        print("         signal. A device-native severity head is viable WITH extra")
        print("         features (device type, event type, hour, recent event mix).")
        print("         A plain lookup table is the floor, not the ceiling: a model")
        print("         with those features should beat this number.")
        print("         PS3 can stay ONE at-failure-time product.")
    else:
        print("      => WEAK even on the impact classes. Subsystem does not determine")
        print("         functional impact. Severity stays ServiceNow-derived and the")
        print("         severity head becomes a POST-WORK-ORDER model -- root cause at")
        print("         failure time, severity later. Two heads, two SLAs.")

    # --- honest caveat ------------------------------------------------------
    print()
    print("      CAVEAT. This is measured only where ServiceNow HAS a record --")
    print(f"      {n6b:,} rows, against 1,059,634 OOS device-days overall. It says")
    print("      subsystem predicts impact ON THE SN-COVERED SUBSET. Whether that")
    print("      generalises to validators (0% SN coverage) is NOT tested here and")
    print("      cannot be, until some validator severity ground truth exists.")
    print("      Do not quote this accuracy for VALIDATOR.")

print("\n" + "=" * 78)
print("Q6b COMPLETE")
print("=" * 78)


In [ ]:
# -----------------------------------------------------------------------------
# VERDICT
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("WHAT TO DO WITH THIS")
print("=" * 78)
print("""  Q1 -- if pct_dev_cover >> pct_sn_cover (especially VALIDATOR), Option 3
        delivers the coverage PS3 has never had. That alone justifies it.
  Q2 -- exact-match >60%: native root cause stands alone, SN becomes audit.
        exact-match <30%: they measure different things -- keep BOTH, and
        treat SN as a second opinion rather than ground truth.
  Q3 -- if device severity orders monotonically with SN failure levels, use
        the device signal and publish the mapping. If not, severity may have
        to stay SN-derived and remain a post-work-order model.
  Q4 -- p90 lag decides whether at-failure-time scoring is possible AT ALL.
        Everything else is moot if the device row lands days later.""")
print("=" * 78)
